# Reproduce the tables and figures

Regenerates every table and figure of the manuscript (`paper/manuscript_v2.md`) from the per-fold results stored in the repository: no EEG data, no GPU and no training are needed. Takes a few minutes.

Colab's preinstalled numpy, pandas, scipy and matplotlib are used. The results agree with the committed ones up to floating-point rounding, which depends on the machine and package versions; the last cell checks this.

In [ ]:
!git clone -q https://github.com/Atytmr07/eeg-fusion-benchmark.git
%cd eeg-fusion-benchmark

## Window-level tables, event tables, controls (Tables 2 and 3)

In [ ]:
!python -m src.make_results --device cuda
!python -m src.controls_stats

In [ ]:
import pandas as pd
pd.read_csv('results_v2/results/controls_chbmit.csv').round(3)

## Figures 2 to 8 and the Supplementary Material

In [ ]:
!python -m src.make_figures --device cuda
!python -m src.make_supplementary

In [ ]:
from IPython.display import Image, display
for n in ['figure2_performance', 'figure3_equivalence', 'figure4_rank_stability',
          'figure5_variance', 'figure6_artefact_removal', 'figure7_seizure_level',
          'figure8_cross_dataset']:
    display(Image(f'paper/figures_manuscript/{n}.png', width=800))

## Check against the committed outputs

Compares every regenerated table with the committed version. Floating-point rounding differs between machines and package versions, so some files are rewritten with different last digits; the check reports the largest relative difference in any value. The figures are regenerated as well; their image files depend on the installed fonts and are not compared.

In [ ]:
import io, subprocess
import numpy as np, pandas as pd

def git(*args):
    return subprocess.run(['git', *args], capture_output=True).stdout

tables = git('ls-files', 'results_v2/results/*.csv').decode().split()
changed = set(git('diff', '--name-only', '--', 'results_v2/results').decode().split())
worst, problems = 0.0, []
for f in tables:
    if f not in changed:
        continue
    old, new = pd.read_csv(io.BytesIO(git('show', f'HEAD:{f}'))), pd.read_csv(f)
    if list(old.columns) != list(new.columns) or old.shape != new.shape:
        problems.append(f)
        continue
    for c in old.columns:
        if pd.api.types.is_numeric_dtype(old[c]) and pd.api.types.is_numeric_dtype(new[c]):
            a, b = old[c].to_numpy(float), new[c].to_numpy(float)
            if not np.array_equal(np.isnan(a), np.isnan(b)):
                problems.append(f'{f}: {c}')
                continue
            ok = ~np.isnan(a)
            if ok.any():
                rel = np.abs(a[ok] - b[ok]) / np.maximum(np.abs(a[ok]), 1e-12)
                worst = max(worst, rel.max())
        elif not old[c].astype(str).equals(new[c].astype(str)):
            problems.append(f'{f}: {c}')
print(f'{len(tables)} committed tables: {len(tables) - len(changed & set(tables))} identical byte for byte, '
      f'{len(changed & set(tables))} rewritten')
print(f'largest relative difference in any value: {worst:.1e}')
print('tables with a different shape, label or missing value:', problems or 'none')